환경 확인하고 4개 csv 불러오기
-customers
-orders


In [ ]:
from pathlib import Path
import sys

# ---- 프로젝트 루트 찾기 (노트북을 어느 폴더에서 열어도 동작한다) ----
# 현재 폴더에서 시작해 위로 한 단계씩 올라가며 data 폴더가 있는 곳을 찾는다.
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "data").is_dir():
    # 최상위(C:\)까지 올라왔는데도 못 찾으면 멈춘다. 이 검사가 없으면 무한 루프가 된다.
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("data 폴더를 찾지 못했습니다.")
    PROJECT_ROOT = PROJECT_ROOT.parent

# course_utils 를 import 할 수 있도록 프로젝트 루트를 검색 경로에 넣는다.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR, "| 존재:", DATA_DIR.exists())

from course_utils.paths import get_project_root, get_data_dir
import pandas as pd

RAW = get_data_dir() / "raw"
print("course_utils 기준 루트:", get_project_root())
print("데이터 폴더:", RAW, "| 존재:", RAW.exists())


In [ ]:
# customers, orders 불러오기
customers = pd.read_csv(RAW / "customers.csv")
orders    = pd.read_csv(RAW / "orders.csv")

print(customers.shape, orders.shape)


In [ ]:
# products, order_items 불러오기
products    = pd.read_csv(RAW / "products.csv")
order_items = pd.read_csv(RAW / "order_items.csv")

print(products.shape, order_items.shape)


In [ ]:
# oreders 의 상위 5개 row만 출력

print(orders.head())

In [ ]:
# 주문 상태의 종류는 몇 가지이고 각 상태별 주문 수량?
ret = orders["order_status"].value_counts(dropna=False)
print(type(ret))
print("상태 종류:", orders["order_status"].nunique(), "가지")
print(ret)

print()

# 결제 수단의 종류는 몇 가지이고 각 결제 수단별 주문 수량?
pay = orders["payment_method"].value_counts(dropna=False)
print("결제 수단 종류:", orders["payment_method"].nunique(), "가지")
print(pay)


## 컬럼 선택 - Series 와 DataFrame


In [ ]:
# 컬럼 하나 -> Series (1차원)
city_series = customers["city"]

# 컬럼 여러 개(리스트) -> DataFrame (2차원)
customer_view = customers[["customer_id", "gender", "age", "city"]]

print(type(city_series))
print(type(customer_view))
print()
print(customer_view.head())


## 단일 조건 필터링 - 불리언 마스크


In [ ]:
# 조건식 자체는 True / False 목록(불리언 마스크)이다
mask = customers["age"] >= 30
print(type(mask))
print(mask.head())
print()

# 마스크를 대괄호에 넣어야 True인 행만 남는다
customers_over_30 = customers[mask]
print("전체:", len(customers), "/ 30세 이상:", len(customers_over_30))
print(customers_over_30.head())


## 복합 조건 - & | ~ (각 조건은 괄호로 감싼다)


In [ ]:
# 30세 이상 "그리고" 서울 거주
seoul_over_30 = customers[
    (customers["age"] >= 30)
    & (customers["city"] == "서울")
]
print("30세 이상 + 서울:", len(seoul_over_30))
print(seoul_over_30.head())


In [ ]:
# 서울 "또는" 부산  -> isin() 이 읽기 쉽다
seoul_or_busan = customers[customers["city"].isin(["서울", "부산"])]
print(seoul_or_busan["city"].value_counts())


In [ ]:
# 완료 주문이 "아닌" 주문  -> 틸드(~)
not_completed = orders[~(orders["order_status"] == "completed")]
print(not_completed["order_status"].value_counts(dropna=False))


## 정렬 - sort_values()


In [ ]:
# 가격이 높은 상품 10개
expensive = products.sort_values("price", ascending=False).head(10)
print(expensive[["product_id", "product_name", "category", "price"]])


In [ ]:
# 카테고리 안에서 가격이 높은 순서 (기준 2개, 방향 각각 지정)
by_cat = products.sort_values(["category", "price"], ascending=[True, False])
print(by_cat[["category", "product_name", "price"]].head(10))


In [ ]:
print(customers["city"].unique())

In [ ]:
# 서울, 부산, 인천 도시의 고객들은 누구인가?
city_customers = customers[customers["city"].isin(["서울", "부산", "인천"])]

print(city_customers.head(10))

In [ ]:
print(city_customers["city"].value_counts())

## 파생 컬럼 - line_total (주문상세 한 행의 금액)


In [ ]:
# 원본을 지키기 위해 작업용 복사본을 만든다
order_items_work = order_items.copy()

order_items_work["line_total"] = (
    order_items_work["quantity"] * order_items_work["unit_price"]
)

print(order_items_work[["order_item_id", "order_id", "quantity", "unit_price", "line_total"]].head())


In [ ]:
# 첫 행을 손으로 검산해서 계산식이 맞는지 확인한다
sample = order_items_work.iloc[0]
expected = sample["quantity"] * sample["unit_price"]
actual = sample["line_total"]
print("수작업:", expected, "/ 파생 컬럼:", actual, "/ 일치:", expected == actual)


## 주문 정보 병합 - order_sales


In [ ]:
# 필요한 컬럼만 골라서 붙인다 (_x, _y 방지)
order_sales = order_items_work.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="order_match",
)

print("병합 전:", len(order_items_work), "-> 병합 후:", len(order_sales))
print(order_sales["order_match"].value_counts(dropna=False))


## 날짜형 변환 - to_datetime


In [ ]:
order_sales_test = order_sales.copy()

print("변환 전 dtype:", order_sales_test["order_date"].dtype)

order_sales_test["order_date"] = pd.to_datetime(
    order_sales_test["order_date"],
    errors="coerce",
)

print("변환 후 dtype:", order_sales_test["order_date"].dtype)
print("order_date에 있는 결측치 수: ", order_sales_test["order_date"].isna().sum())


## coerce 동작 확인 - 없는 날짜를 넣어보기


In [ ]:
order_sales_test2 = order_sales.copy()

# 2026-02-30 은 존재하지 않는 날짜다
order_sales_test2.loc[0, "order_date"] = "2026-02-30"

order_sales_test2["order_date"] = pd.to_datetime(
    order_sales_test2["order_date"],
    errors="coerce",
)

print(order_sales_test2.info())
print(order_sales_test2.head(5))
print("order_date에 있는 결측치 수: ", order_sales_test2["order_date"].isna().sum())


In [ ]:
order_sales_test2.head(5)

In [ ]:
order_sales_test2 = order_sales_test2.dropna()
print(order_sales_test2["order_date"].isna().sum())

In [ ]:
# 파생컬럼 order_month 추가하고 0으로 초기화
order_sales_test2["order_month"] = order_sales_test2["order_date"].dt.to_period("M").astype(str) # 연-월 형태로 변환이므로 str로 변환
print(order_sales_test2.head())